In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import os
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt


In [ ]:
img_dir = os.path.join(path, "dataset", "images")
mask_dir = os.path.join(path, "dataset", "masks")

print("img files:", len(os.listdir(img_dir)))
print("mask files:", len(os.listdir(mask_dir)))

In [ ]:
def make_pairs(img_dir, mask_dir):
    imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(".jpg")]
    masks = [f for f in os.listdir(mask_dir) if f.lower().endswith((".png", ".jpg"))]

    mask_map = {}
    for m in masks:
        stem = os.path.splitext(m)[0]
        mask_map[stem] = m

    pairs = []
    for im in imgs:
        stem = os.path.splitext(im)[0]
        if stem in mask_map:
            pairs.append((im, mask_map[stem]))
    return sorted(pairs)

class SUIMDataset(Dataset):
    def __init__(self, img_dir, mask_dir, size=(256,256)):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.size = size
        self.pairs = make_pairs(img_dir, mask_dir)
        print("num matched pairs:", len(self.pairs))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_name, mask_name = self.pairs[idx]

        img = torchvision.io.read_image(os.path.join(self.img_dir, img_name)).float() / 255.0  # (3,H,W)
        mask = torchvision.io.read_image(os.path.join(self.mask_dir, mask_name))                # (C,H,W)

        if mask.size(0) > 1:
            mask = mask[0:1]  # take first channel
        mask = mask.long().squeeze(0)  # (H,W)

        img = F.interpolate(img.unsqueeze(0), size=self.size, mode="bilinear", align_corners=False).squeeze(0)
        mask = F.interpolate(mask.unsqueeze(0).unsqueeze(0).float(), size=self.size, mode="nearest").squeeze(0).squeeze(0).long()

        mask = remap_mask(mask)
        return img, mask

In [ ]:
ds = SUIMDataset(img_dir, mask_dir, size=(256,256))

train_len = int(0.8 * len(ds))
val_len = len(ds) - train_len
train_ds, val_ds = random_split(ds, [train_len, val_len])

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=0)

In [ ]:
imgs, masks = next(iter(train_loader))

plt.figure(figsize=(10,6))
for i in range(2):
    plt.subplot(2,2,i*2+1)
    plt.imshow(imgs[i].permute(1,2,0))
    plt.title("Image")
    plt.axis("off")

    plt.subplot(2,2,i*2+2)
    plt.imshow(masks[i], cmap="tab20", vmin=0, vmax=7)
    plt.title("Mask")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!pip -q install segmentation-models-pytorch

In [ ]:
# TO DO
import segmentation_models_pytorch as smp

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8
).to(device)

In [ ]:
# TO DO
def train_one_epoch_seg(model, loader, loss_fn, opt, device):
    model.train()
    total_loss, total = 0, 0

    for imgs, masks in loader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        opt.zero_grad()
        out = model(imgs)
        loss = loss_fn(out, masks)
        loss.backward()
        opt.step()

        total_loss += loss.item() * imgs.size(0)
        total += imgs.size(0)

    return total_loss / total

In [ ]:
def validate_one_epoch_seg(model, loader, loss_fn, device):
    model.eval()
    total_loss, total = 0, 0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            masks = masks.to(device)

            out = model(imgs)
            loss = loss_fn(out, masks)

            total_loss += loss.item() * imgs.size(0)
            total += imgs.size(0)

    return total_loss / total

In [ ]:
# TO DO
loss_fn = torch.nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
train_losses, val_losses = [], []

for epoch in range(epochs):
    tr = train_one_epoch_seg(model, train_loader, loss_fn, opt, device)
    va = validate_one_epoch_seg(model, val_loader, loss_fn, device)

    train_losses.append(tr)
    val_losses.append(va)

    print(f"Epoch {epoch+1}/{epochs} | train loss {tr:.4f} | val loss {va:.4f}")

In [ ]:
plt.figure()
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch"); plt.ylabel("loss")
plt.legend(); plt.show()

In [ ]:
# TO DO
model.eval()

imgs, masks = next(iter(val_loader))
imgs = imgs.to(device)

with torch.no_grad():
    out = model(imgs)
    preds = out.argmax(1).cpu()

imgs = imgs.cpu()
masks = masks.cpu()

n = min(3, imgs.size(0))

plt.figure(figsize=(12, 4*n))
for i in range(n):
    plt.subplot(n,3,i*3+1)
    plt.imshow(imgs[i].permute(1,2,0))
    plt.title("Image")
    plt.axis("off")

    plt.subplot(n,3,i*3+2)
    plt.imshow(masks[i], cmap="tab20", vmin=0, vmax=7)
    plt.title("GT")
    plt.axis("off")

    plt.subplot(n,3,i*3+3)
    plt.imshow(preds[i], cmap="tab20", vmin=0, vmax=7)
    plt.title("Pred")
    plt.axis("off")

plt.tight_layout()
plt.show()